In [80]:
# To run this script, you will need data populated in data/parameter_sweeps/synthesis. You can do so by running the "run_stellar_host_mass_parameter_sweeps.py" script.

import numpy as np
import os
import glob
from paths import path
from lib import read_vplanet

In [81]:
# Computes the HZ limits from Kopparapu et al. (2014)

RECENT_VENUS = 0
RUNAWAY_GREENHOUSE = 1
MAXIMUM_GREENHOUSE = 2
EARLY_MARS = 3

# For 1 Earth mass.
# Column 1: Recent venus
# Column 2: Runaway greenhouse
# Column 3: Maximum greenhouse
# Column 4: Early mars
seff_solar_2014 = np.array([1.776, 1.107, 0.356, 0.32])

coeff_2014 = np.array([
    # Column 1: coefficient a
    # Column 2: coefficient b
    # Column 3: coefficient c
    # Column 4: coefficient d
    [2.136e-04,  2.533e-08, -1.332e-11, -3.097e-15], # Row 1: Recent venus
    [1.332e-04,  1.580e-08, -8.308e-12, -1.931e-15], # Row 2: Runaway greenhouse
    [6.171e-05,  1.698e-09, -3.198e-12, -5.575e-16], # Row 3: Maximum greenhouse
    [5.547e-05,  1.526e-09, -2.874e-12, -5.011e-16], # Row 4: Early mars
])

def get_distance_2014_AU(luminosity_LSun, teff_K):
    tstar_K = teff_K - 5780 # Kelvin.
    tvector_K = np.array([tstar_K, tstar_K**2, tstar_K**3, tstar_K**4])

    seff_2014 = seff_solar_2014 + np.matmul(coeff_2014, tvector_K)

    return np.sqrt(luminosity_LSun/seff_2014)

In [99]:
for dir in sorted(glob.glob(path("data", "parameter_sweeps", "synthesis", "*"))):
    # Skip the notebook file.
    if '.' in dir or '_water' in dir:
        continue

    out = read_vplanet(path(dir, "run_K0_ThU0_a0", "star.star.forward"), output_options = ["Age", "Luminosity", "Temperature"], usecols = [1, 2, 5])

    time_yr = out["Age"]
    mask = time_yr == 4.6e9

    temperature_K = out["Temperature"][mask].iloc[0]
    luminosity_LSun = out["Luminosity"][mask].iloc[0]

    distances_AU = get_distance_2014_AU(luminosity_LSun, temperature_K)
    inner_HZ_AU = distances_AU[RUNAWAY_GREENHOUSE] # Runaway greenhouse.
    outer_HZ_AU = distances_AU[MAXIMUM_GREENHOUSE] # Maximum greenhouse.

    stellar_label = dir.split(os.sep)[-1].replace("_atm", "")

    output = 'HZ for {d} is: {a}-{b} AU'.format(d = stellar_label, a = inner_HZ_AU, b = outer_HZ_AU)

    print(output)

HZ for kv is: 0.6401830775621882-1.1510924277633037 AU
HZ for m0p1 is: 0.031060592538580522-0.06229305781670558 AU
HZ for m0p2 is: 0.07235421109392776-0.1421672474441826 AU
HZ for m0p3 is: 0.10877635393888095-0.2121318467235588 AU
HZ for m0p4 is: 0.14612964266104495-0.28349597780127755 AU
HZ for m0p5 is: 0.19761127501413284-0.3802637544815201 AU
HZ for m0p6 is: 0.2776971625930472-0.5262273650739901 AU
HZ for m0p7 is: 0.39124504606264093-0.7254171525991292 AU
HZ for m0p8 is: 0.5359642927048134-0.9735022355856436 AU
HZ for m0p9 is: 0.7194908169548471-1.2859924765981463 AU
HZ for m1p1 is: 1.2700977314900757-2.2277696678863275 AU
HZ for m1p2 is: 1.5146762666232594-2.6475155997806903 AU
HZ for sun is: 0.9627690052874481-1.6998823971640933 AU
HZ for trappist is: 0.024967375866069036-0.050430828529552864 AU


In [100]:
trappist_central_HZ = (0.024967375866069036 + 0.050430828529552864)/2
trappist_central_HZ

0.037699102197810946

In [101]:
sun_central_HZ = (0.9627690052874481 + 1.6998823971640933)/2
sun_central_HZ

1.3313257012257707

In [102]:
kv_central_HZ = (0.6401830775621882 + 1.1510924277633037)/2
kv_central_HZ

0.8956377526627459

In [93]:
parameter_sweeps_directory = glob.glob(path("data", "parameter_sweeps", "*"))

for sweep in parameter_sweeps_directory:
    # Skip files, seek relevant directories only.
    if "__" in sweep or "." in sweep or "synthesis" in sweep:
        continue

    run = path(sweep, "run_K0_ThU0")

    for file in os.listdir(run):
        print(file)

        if ".forward" in file and not "earth" in file and not "water" in file:            
            out = read_vplanet(path(run, file))

            time_yr = out["Age"]
            mask = time_yr == 4.6e9

            temperature_K = out["Temperature"][mask].iloc[0]
            luminosity_LSun = out["Luminosity"][mask].iloc[0]

            distances_AU = get_distance_2014_AU(luminosity_LSun, temperature_K)

            inner_HZ_AU = distances_AU[RUNAWAY_GREENHOUSE]
            outer_HZ_AU = distances_AU[MAXIMUM_GREENHOUSE]

            central_HZ_AU = (inner_HZ_AU + outer_HZ_AU) / 2.0

            output = "HZ for {d} is: {a}-{b} AU. Central HZ is {c}".format(d = file, a = inner_HZ_AU, b = outer_HZ_AU, c = central_HZ_AU)

            print(output)

earth.in
sun.in
vpl.in
vplanet_log
earth.in
trappist.earth.forward
trappist.trappist_1.forward
HZ for trappist.trappist_1.forward is: 0.024967375866069036-0.050430828529552864 AU. Central HZ is 0.037699102197810946
trappist_1.in
vpl.in
trappist.log
sol.sun.forward
HZ for sol.sun.forward is: 0.9627690052874481-1.6998823971640933 AU. Central HZ is 1.3313257012257707
earth.in
sun.in
sol.log
vpl.in
vplanet_log
sol.earth.forward
earth.in
kv.log
kv.in
kv.earth.forward
kv.kv.forward
HZ for kv.kv.forward is: 0.6401830775621882-1.1510924277633037 AU. Central HZ is 0.8956377526627459
vpl.in
